# A Clean Place is a Safe Place

This notebook prepares the KKBox churn data. One output row is one target `msno` at a
cutoff. There are **two labelled monthly cohorts** and one unlabelled test cohort:

| Cohort | Feature cutoff | Label month | Label source | Role in w1 models |
|---|---|---|---|---|
| `train_feb` | 2017-01-31 | February 2017 | `train.csv` (v1) | fit / stop / tune |
| `train` | 2017-02-28 | March 2017 | `train_v2.csv` | cross-month holdout |
| `test` | 2017-03-31 | April 2017 | `sample_submission_v2.csv` | final scoring |

The two labelled cohorts allow a month-over-month validation: models learn on February and are
judged on March, which mimics predicting April from a model trained on earlier months.
All transformation code is here.

**Route:** raw archives → target users → members, transactions and daily listening →
user features → audited v4 tabular export (`export-tabular-v4/`).

| Product | Shape / grain | Consumer |
|---|---|---|
| `train_feb.parquet`, `train.parquet`, `test.parquet` | one row per cohort user | tabular models |
| `metadata.json`, `preprocess_params.json` | one file per export | contract and audit |

Training, Optuna and model metrics live in `w1_tabular_models.ipynb`.

**Contents:** [configuration](#1-configuration) · [sources](#2-raw-sources-and-parquet) ·
[users](#3-target-users-and-calendar) · [members](#4-members) ·
[transactions](#5-transactions-and-deduplication) · [aggregates](#6-transaction-features) ·
[listening](#7-daily-listening) · [parameters](#8-listening-features-and-parameters) ·
[joins](#9-final-user-table) · [export](#10-tabular-export).


## 1. Configuration

`KKBOX_SAMPLE_USERS=100` runs a small, isolated export in `export-tabular-v4-sample/` (the first
N users of each cohort). The normal run uses all users and writes `export-tabular-v4/`.
The three cutoffs are the v4 contract. Reusing the v2 export is not supported here because v2
has no February cohort. Earlier exports (`export-tabular-v3/`) are left untouched, so the w2
notebooks keep working on them.


In [9]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess

import numpy as np
import pandas as pd
import polars as pl


LOG_COLS = ["num_25", "num_50", "num_75", "num_985", "num_100", "num_unq", "total_secs"]
LOG_NAMES = {
    "num_25": "25", "num_50": "50", "num_75": "75", "num_985": "985",
    "num_100": "100", "num_unq": "unq", "total_secs": "total_secs",
}

CATEGORICAL = ["city", "registered_via", "last_payment_method_id"]

SEQ_DAYS = 90
SEQ_CHANNELS = LOG_COLS + ["active"]

# Two labelled monthly cohorts plus the unlabelled April test cohort.
# train_feb: train.csv (v1) labels February churn from history through January 31.
# train:     train_v2.csv labels March churn from history through February 28.
# test:      sample_submission_v2.csv is scored for April from history through March 31.
CUTOFF = {"train_feb": "2017-01-31", "train": "2017-02-28", "test": "2017-03-31"}
LABEL_MONTHS = {"train_feb": "2017-02", "train": "2017-03", "test": "2017-04"}
LABEL_FILES = {"train_feb": "train", "train": "train_v2"}   # cohort -> labelled parquet
SPLITS = ["train_feb", "train", "test"]
DATASET_VERSION = 4

_TXN_KEYS = ["payment_method_id", "payment_plan_days", "plan_list_price", "actual_amount_paid",
             "is_auto_renew", "transaction_date", "membership_expire_date", "is_cancel"]

# Explicit schemas for the CSV -> parquet conversion (plan 0.5). Smallest types that fit the
# observed ranges (CP-01): city 1-22, registered_via -1-19, bd -7168-2016, plan_days 0-450,
# prices 0-2000, payment_method_id 2-41.
SCHEMA = {
    "train": {"msno": pl.String, "is_churn": pl.Int8},
    "train_v2": {"msno": pl.String, "is_churn": pl.Int8},
    "sample_submission_v2": {"msno": pl.String, "is_churn": pl.Int8},
    "members_v3": {"msno": pl.String, "city": pl.Int8, "bd": pl.Int16,
                   "registered_via": pl.Int8, "registration_init_time": pl.Int32,
                   "gender": pl.String},
    "transactions": {"msno": pl.String, "payment_method_id": pl.Int8, "payment_plan_days": pl.Int16,
                     "plan_list_price": pl.Int16, "actual_amount_paid": pl.Int16,
                     "is_auto_renew": pl.Int8, "transaction_date": pl.Int32,
                     "membership_expire_date": pl.Int32, "is_cancel": pl.Int8},
    "transactions_v2": {"msno": pl.String, "payment_method_id": pl.Int8,
                        "payment_plan_days": pl.Int16, "plan_list_price": pl.Int16,
                        "actual_amount_paid": pl.Int16, "is_auto_renew": pl.Int8,
                        "transaction_date": pl.Int32, "membership_expire_date": pl.Int32,
                        "is_cancel": pl.Int8},
    "user_logs": {"msno": pl.String, "date": pl.Int32, "num_25": pl.Int32, "num_50": pl.Int32,
                  "num_75": pl.Int32, "num_985": pl.Int32, "num_100": pl.Int32,
                  "num_unq": pl.Int32, "total_secs": pl.Float64},
    "user_logs_v2": {"msno": pl.String, "date": pl.Int32, "num_25": pl.Int32, "num_50": pl.Int32,
                     "num_75": pl.Int32, "num_985": pl.Int32, "num_100": pl.Int32,
                     "num_unq": pl.Int32, "total_secs": pl.Float64},
}

_COLUMNS = {
    "train": ["msno", "is_churn"],
    "train_v2": ["msno", "is_churn"],
    "sample_submission_v2": ["msno"],
    "members_v3": ["msno", "city", "registered_via", "bd", "registration_init_time", "gender"],
    "transactions": ["msno"] + _TXN_KEYS,
    "transactions_v2": ["msno"] + _TXN_KEYS,
    "user_logs": ["msno", "date"] + LOG_COLS,
    "user_logs_v2": ["msno", "date"] + LOG_COLS,
}

In [10]:
import gc
import os
import resource

DATA_DIR = Path('../data')
PARQUET_DIR = DATA_DIR / 'parquet'
SAMPLE_USERS = int(os.environ['KKBOX_SAMPLE_USERS']) if os.environ.get('KKBOX_SAMPLE_USERS') else None
if SAMPLE_USERS is not None and SAMPLE_USERS < 1:
    raise ValueError('KKBOX_SAMPLE_USERS must be positive.')
if os.environ.get('KKBOX_REUSE_V2', '0') == '1':
    raise ValueError('KKBOX_REUSE_V2 is not supported by the v4 export (v2 has no February cohort).')
EXPORT_DIR = Path('export-tabular-v4-sample' if SAMPLE_USERS is not None else 'export-tabular-v4')
if shutil.disk_usage(EXPORT_DIR.resolve().parent).free < 2_000_000_000:
    raise OSError('Insufficient free disk for the tabular export (~2 GB).')
print('Building member, transaction and listening features from parquet.')
print('Export directory:', EXPORT_DIR, '| sample users per cohort:', SAMPLE_USERS)


Building member, transaction and listening features from parquet.
Export directory: export-tabular-v4 | sample users per cohort: None


## 2. Raw sources and parquet

| CSV archive | Grain / key | Fields used |
|---|---|---|
| `train` | February-labelled user, `msno` | `is_churn` |
| `train_v2` | March-labelled user, `msno` | `is_churn` |
| `sample_submission_v2` | score user, `msno` | identifier only |
| `members_v3` | member, `msno` | city, age, registration source/date |
| `transactions`, `transactions_v2` | payment record, `msno` and transaction fields | plans, amounts, dates, cancellation, renewal |
| `user_logs`, `user_logs_v2` | listening record, `msno` and date | seven listening measures |

The two transaction and listening releases jointly cover the three cutoffs.
Conversion selects used columns and assigns compact physical types; it does not
impute or clip values. Existing parquet files are skipped. Extraction handles one
archive at a time and materializes parquet at `sink_parquet`; no conversion occurs
until the following execution cell.

In [11]:
def window_start(cutoff):
    """First day of the 3-month window that ends with the cutoff month."""
    return (pd.Timestamp(cutoff).to_period("M") - 2).start_time


def sequence_start(cutoff):
    """First of the 90 days that end on the cutoff."""
    return pd.Timestamp(cutoff) - pd.Timedelta(days=SEQ_DAYS - 1)


def _date_int(value):
    return int(pd.Timestamp(value).strftime("%Y%m%d"))


def _date(value):
    return pd.Timestamp(value).date()


def _7z_binary():
    for binary in ("7z", "7za", "7zz"):
        path = shutil.which(binary)
        if path:
            return path
    raise RuntimeError("Converting .7z files needs the `7z` command (p7zip).")

In [12]:
def convert_raw_to_parquet(data_dir, parquet_dir=None, archives=None):
    """Convert ``data/*.csv.7z`` to ``data/parquet/*.parquet`` (zstd), idempotently.

    Each archive is extracted to ``data/_extract`` one at a time, streamed to parquet and the
    CSV deleted, so the disk peak is one extracted CSV plus the parquet being written.
    """
    data_dir = Path(data_dir)
    parquet_dir = Path(parquet_dir) if parquet_dir else data_dir / "parquet"
    parquet_dir.mkdir(parents=True, exist_ok=True)
    extract_dir = data_dir / "_extract"

    for name in (archives or list(SCHEMA)):
        out = parquet_dir / f"{name}.parquet"
        if out.exists():
            print(f"[skip] {out.name} already exists")
            continue
        archive = data_dir / f"{name}.csv.7z"
        if not archive.exists():
            print(f"[skip] {archive} not found")
            continue

        print(f"[extract] {archive.name}")
        subprocess.run([_7z_binary(), "x", "-y", f"-o{extract_dir}", str(archive)],
                       check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        csv = next(path for path in extract_dir.rglob(f"{name}.csv"))
        print(f"[convert] {name}.csv -> {out.name}")
        (pl.scan_csv(str(csv), schema_overrides=SCHEMA[name])
           .select(_COLUMNS[name])
           .sink_parquet(out, compression="zstd"))
        csv.unlink()

        if name.startswith("user_logs"):
            rows = pl.scan_parquet(out).select(pl.len()).collect().item()
            print(f"          {rows:,} rows")

    return parquet_dir

In [13]:
convert_raw_to_parquet(DATA_DIR)
print('Parquet files:', [p.name for p in sorted(PARQUET_DIR.glob('*.parquet'))])
print('Train schema:', pl.scan_parquet(PARQUET_DIR / 'train_v2.parquet').collect_schema())

[skip] train.parquet already exists
[skip] train_v2.parquet already exists
[skip] sample_submission_v2.parquet already exists
[skip] members_v3.parquet already exists
[skip] transactions.parquet already exists
[skip] transactions_v2.parquet already exists
[skip] user_logs.parquet already exists
[skip] user_logs_v2.parquet already exists
Parquet files: ['members_v3.parquet', 'sample_submission_v2.parquet', 'train.parquet', 'train_v2.parquet', 'transactions.parquet', 'transactions_v2.parquet', 'user_logs.parquet', 'user_logs_v2.parquet']
Train schema: Schema({'msno': String, 'is_churn': Int8})


## 3. Target users and calendar

| Cohort | Monthly window (3 months) | Known through | Target month |
|---|---|---|---|
| `train_feb` | 2016-11-01 to 2017-01-31 | 2017-01-31 | February 2017 |
| `train` | 2016-12-01 to 2017-02-28 | 2017-02-28 | March 2017 |
| `test` | 2017-01-01 to 2017-03-31 | 2017-03-31 | April 2017 |

Monthly aggregates use calendar months. `msno` is the source identifier, `uid` is a code
shared across the three cohorts, and `row` is the position within one cohort's source file.
Most users appear in several cohorts (each time with its own cutoff, features and label);
that is expected and mirrors production, where the same subscriber is scored every month.
The source file's order determines output rows.

**Label source.** The February labels are the official `train.csv`. A Polars port of
`data/WSDMChurnLabeller.scala` was checked against both official files: it agrees on
≈ 98.8 % (February) and ≈ 97.0 % (March) of the shared users, but about 137k official users
per month cannot be reproduced from the published transactions. The official labels are
therefore used as given for both months. Their prevalence differs (≈ 6.4 % in February,
≈ 9.0 % in March); that shift is part of what the cross-month holdout measures.


In [14]:
def load_target_users(parquet_dir):
    """Load the labelled cohorts and test users and build the ``msno`` -> ``uid`` union index.

    Returns ``(labelled, test, user_index)`` where ``labelled`` maps cohort name -> frame with
    ``msno``, ``is_churn`` and ``row``. ``uid`` is assigned over the union of all cohorts;
    ``<cohort>_row`` holds the position of the user in that cohort's source file (null when
    absent), because the competition files overlap.
    """
    parquet_dir = Path(parquet_dir)
    labelled = {name: (pl.read_parquet(parquet_dir / f"{file}.parquet", columns=["msno", "is_churn"])
                       .with_row_index("row"))
                for name, file in LABEL_FILES.items()}
    test = (pl.read_parquet(parquet_dir / "sample_submission_v2.parquet", columns=["msno"])
            .with_row_index("row"))

    cohorts = {**labelled, "test": test}
    union = pl.concat([frame.select("msno") for frame in cohorts.values()]).unique(maintain_order=True)
    user_index = (union.with_row_index("uid")
                  .with_columns(pl.col("uid").cast(pl.UInt32)))
    for name, frame in cohorts.items():
        user_index = user_index.join(
            frame.select("msno", pl.col("row").alias(f"{name}_row")), on="msno", how="left",
            maintain_order="left")
    return labelled, test, user_index


def users_with_codes(target_users, user_index):
    """Return ``target_users`` as ``msno`` + ``uid`` + ``row`` in the original order."""
    return (target_users.select("msno", "row")
            .join(user_index.select("msno", "uid"), on="msno", how="left", maintain_order="left"))


In [15]:
def _hash_msno(df):
    digest = hashlib.sha1()
    for value in df["msno"].to_list():
        digest.update(str(value).encode())
        digest.update(b"\n")
    return digest.hexdigest()




In [16]:
labelled_users, test_users, user_index = load_target_users(PARQUET_DIR)
if SAMPLE_USERS is not None:
    labelled_users = {name: frame.head(SAMPLE_USERS) for name, frame in labelled_users.items()}
    test_users = test_users.head(SAMPLE_USERS)
    selected = pl.concat([frame.select('msno') for frame in [*labelled_users.values(), test_users]]).unique()
    user_index = user_index.join(selected, on='msno', how='semi')
source_users = {**labelled_users, 'test': test_users}
users_by_split = {name: users_with_codes(source_users[name], user_index) for name in SPLITS}
for name in SPLITS:
    assert source_users[name]['msno'].is_unique().all(), f'{name} has duplicated msno'
    churn = (f"| churn rate: {labelled_users[name]['is_churn'].mean():.4f}"
             if name in labelled_users else '| unlabelled')
    print(f'{name:9s} users: {source_users[name].height:>9,}', churn)
print('unique uid over all cohorts:', user_index.height)
print('train_feb users also in March cohort:',
      users_by_split['train_feb'].join(users_by_split['train'], on='uid', how='semi').height)
for split, cutoff in CUTOFF.items():
    print(split, 'monthly start:', window_start(cutoff).date(), 'cutoff:', cutoff,
          'label month:', LABEL_MONTHS[split])


train_feb users:   992,931 | churn rate: 0.0639
train     users:   970,960 | churn rate: 0.0899
test      users:   907,471 | unlabelled
unique uid over all cohorts: 1143472
train_feb users also in March cohort: 881701
train_feb monthly start: 2016-11-01 cutoff: 2017-01-31 label month: 2017-02
train monthly start: 2016-12-01 cutoff: 2017-02-28 label month: 2017-03
test monthly start: 2017-01-01 cutoff: 2017-03-31 label month: 2017-04


## 4. Members

Members are filtered to target `uid`s before building features. Valid age is
1–100; other ages receive the mean valid train age and `bd_missing=1`.
Negative `registered_via` receives the train mode. Registration dates yield
tenure, year and month at each cutoff. `gender` is loaded by the raw schema but
excluded from the existing feature catalog. These imputation values are fitted
once on the February cohort (the training cohort), never on the March holdout or test.

In [17]:
def scan_members(parquet_dir, user_index):
    """Lazy members table filtered to target users, keyed by ``uid``."""
    return (pl.scan_parquet(Path(parquet_dir) / "members_v3.parquet")
            .join(user_index.lazy().select("msno", "uid"), on="msno", how="inner")
            .select("uid", "city", "registered_via", "bd", "registration_init_time"))


def scan_transactions(parquet_dir, user_index, max_date):
    """Lazy transactions (both files, deduplicated) up to ``max_date``, keyed by ``uid``."""
    parquet_dir = Path(parquet_dir)
    frames = [pl.scan_parquet(parquet_dir / "transactions.parquet"),
              pl.scan_parquet(parquet_dir / "transactions_v2.parquet")]
    return (pl.concat(frames)
            .with_row_index("_ord")
            .join(user_index.lazy().select("msno", "uid"), on="msno", how="inner",
                  maintain_order="left")
            .filter(pl.col("transaction_date") <= _date_int(max_date))
            .unique(subset=["msno", *_TXN_KEYS], keep="first", maintain_order=True)
            .select("uid", "_ord", *_TXN_KEYS))

In [18]:
def build_member_features(members, users, params, cutoff):
    """Clean members and build static features (MOD-01)."""
    reg = pl.col("registration_init_time").cast(pl.String).str.to_date("%Y%m%d")
    return (members.join(users.select("uid"), on="uid", how="semi")
            .select(
                pl.col("uid"),
                pl.col("city"),
                pl.when(pl.col("registered_via") >= 0).then(pl.col("registered_via"))
                  .otherwise(params["registered_via_mode"]).cast(pl.Int16).alias("registered_via"),
                pl.when((pl.col("bd") > 0) & (pl.col("bd") <= 100)).then(pl.col("bd"))
                  .otherwise(params["age_mean"]).cast(pl.Float32).alias("bd"),
                pl.when((pl.col("bd") <= 0) | (pl.col("bd") > 100)).then(1).otherwise(0)
                  .cast(pl.Int8).alias("bd_missing"),
                (pl.lit(_date(cutoff)) - reg).dt.total_days().cast(pl.Int32).alias("tenure_days"),
                reg.dt.year().cast(pl.Int16).alias("registration_year"),
                reg.dt.month().cast(pl.Int8).alias("registration_month"),
            ))

## 5. Transactions and deduplication

Both releases are concatenated, filtered to target users and cutoff, then exact
duplicates are removed using `msno` plus all eight transaction fields. The first
copy is kept. Two records on the same day with different values remain separate.
`_ord` records stable source order; it is a tie break, **not an intraday time**.
Conflicting cancellation, renewal or expiry values on one day mark ambiguous
state. `is_cancel=1` is a payment record field, not the churn target label.

The sequence preflight rejects missing/invalid required transaction dates, plan
days and binary flags; missing expiry has a separate mask. It counts negative
amounts and invalid expiry values for the audit. A future transaction never
updates an earlier day's state.

In [19]:
members = scan_members(PARQUET_DIR, user_index).collect(engine='streaming')
transactions = scan_transactions(PARQUET_DIR, user_index, CUTOFF['test']).collect(engine='streaming')
print('Members:', members.shape, '| transactions:', transactions.shape)
print(transactions.select('uid', '_ord', 'transaction_date', 'actual_amount_paid', 'is_cancel').head(3))

Members: (1018235, 5) | transactions: (17858127, 10)
shape: (3, 5)
┌────────┬──────┬──────────────────┬────────────────────┬───────────┐
│ uid    ┆ _ord ┆ transaction_date ┆ actual_amount_paid ┆ is_cancel │
│ ---    ┆ ---  ┆ ---              ┆ ---                ┆ ---       │
│ u32    ┆ u32  ┆ i32              ┆ i16                ┆ i8        │
╞════════╪══════╪══════════════════╪════════════════════╪═══════════╡
│ 110581 ┆ 0    ┆ 20150930         ┆ 129                ┆ 0         │
│ 724815 ┆ 1    ┆ 20150930         ┆ 149                ┆ 0         │
│ 789025 ┆ 4    ┆ 20150930         ┆ 149                ┆ 0         │
└────────┴──────┴──────────────────┴────────────────────┴───────────┘


In [20]:
# Two different payments on one date survive; an exact second copy does not.
first_transaction = transactions.row(0, named=True)
duplicate_example = pl.DataFrame([
    first_transaction,
    dict(first_transaction),
    {**first_transaction, 'actual_amount_paid': first_transaction['actual_amount_paid'] + 1},
])
kept_example = duplicate_example.unique(subset=['uid', *_TXN_KEYS], keep='first')
print('Same-day source rows:', duplicate_example.height, '| after exact deduplication:', kept_example.height)
print(kept_example.select('uid', 'transaction_date', 'actual_amount_paid'))

Same-day source rows: 3 | after exact deduplication: 2
shape: (2, 3)
┌────────┬──────────────────┬────────────────────┐
│ uid    ┆ transaction_date ┆ actual_amount_paid │
│ ---    ┆ ---              ┆ ---                │
│ i64    ┆ i64              ┆ i64                │
╞════════╪══════════════════╪════════════════════╡
│ 110581 ┆ 20150930         ┆ 129                │
│ 110581 ┆ 20150930         ┆ 130                │
└────────┴──────────────────┴────────────────────┘


## 6. Transaction features

Tabular transaction features use **all known history up to the cutoff**, including
records before the 90-day tensor. They include counts and sums, plan/amount means
and distinct counts, renewal/cancellation summaries, the last record, and recency.
`duration = expiry - transaction date` and `plan_coverage_ratio = duration / plan
days` when plan days are positive. The last record sorts by transaction date then
`_ord`; `days_to_expire = last expiry - cutoff`, so it can be negative. A user with
no transactions gets nulls until the final left join fills feature nulls with zero.

In [21]:
def build_transaction_features(trans, users, cutoff):
    """Aggregate transactions per user up to the cutoff (MOD-02)."""
    cutoff_int = _date_int(cutoff)
    t = ((trans.lazy() if isinstance(trans, pl.DataFrame) else trans)
         .join(users.lazy().select("uid"), on="uid", how="semi")
         .filter(pl.col("transaction_date") <= cutoff_int)
         .with_columns(
             pl.col("transaction_date").cast(pl.String).str.to_date("%Y%m%d").alias("td"),
             pl.col("membership_expire_date").cast(pl.String).str.to_date("%Y%m%d").alias("ed")))

    t = t.with_columns((pl.col("ed") - pl.col("td")).dt.total_days().cast(pl.Int32).alias("duration"))
    t = t.with_columns(
        (pl.col("duration") / pl.when(pl.col("payment_plan_days") > 0)
            .then(pl.col("payment_plan_days")).otherwise(None)).alias("plan_coverage_ratio"),
        (pl.col("duration") > 30).cast(pl.Int8).alias("more_than_30"),
        ((pl.col("is_auto_renew") == 1) & (pl.col("is_cancel") == 0)).cast(pl.Int8)
            .alias("autorenew_not_cancel"),
        ((pl.col("is_auto_renew") == 0) & (pl.col("is_cancel") == 1)).cast(pl.Int8)
            .alias("not_autorenew_cancel"),
        (pl.col("plan_list_price") - pl.col("actual_amount_paid")).cast(pl.Int16)
            .alias("discount"))

    last = lambda col: pl.col(col).sort_by([pl.col("td"), pl.col("_ord")]).last()  # noqa: E731
    last_td, last_ed = last("td"), last("ed")

    return t.group_by("uid").agg(
        pl.col("is_cancel").max().alias("is_cancel_change_flag"),
        pl.col("is_auto_renew").max().alias("is_auto_renew_change_flag"),
        pl.col("payment_method_id").count().alias("transaction_count"),
        pl.col("actual_amount_paid").mean().alias("average_amount_paid"),
        pl.col("plan_list_price").mean().alias("average_amount_charged"),
        pl.col("payment_plan_days").mean().alias("average_plan"),
        pl.col("payment_method_id").n_unique().alias("n_payment_methods"),
        pl.col("payment_plan_days").n_unique().alias("n_plans"),
        pl.col("plan_list_price").n_unique().alias("n_list_prices"),
        pl.col("actual_amount_paid").n_unique().alias("n_amounts_paid"),
        pl.col("duration").mean().alias("duration"),
        pl.col("plan_coverage_ratio").mean().alias("plan_coverage_ratio"),
        pl.col("payment_plan_days").sum().alias("sum_plan_days"),
        pl.col("plan_list_price").sum().alias("sum_list_price"),
        pl.col("actual_amount_paid").sum().alias("sum_amount_paid"),
        pl.col("more_than_30").sum().alias("more_than_30"),
        pl.col("autorenew_not_cancel").min().alias("autorenew_not_cancel"),
        pl.col("not_autorenew_cancel").min().alias("not_autorenew_cancel"),
        pl.col("payment_plan_days").max().alias("max_plan_days"),
        last("payment_method_id").cast(pl.Int8).alias("last_payment_method_id"),
        last("payment_plan_days").cast(pl.Int16).alias("last_plan_days"),
        last("is_auto_renew").cast(pl.Int8).alias("last_is_auto_renew"),
        last("is_cancel").cast(pl.Int8).alias("last_is_cancel"),
        last("discount").alias("last_discount"),
        (last_ed - pl.lit(_date(cutoff))).dt.total_days().cast(pl.Int32).alias("days_to_expire"),
        (last_ed.dt.weekday() - 1).cast(pl.Int8).alias("expire_dayofweek"),
        (pl.lit(_date(cutoff)) - last_td).dt.total_days().cast(pl.Int32)
            .alias("days_since_last_transaction"),
    ).with_columns(
        (pl.col("average_amount_paid") - pl.col("average_amount_charged")).alias("difference")
    ).collect(engine="streaming")

## 7. Daily listening

The two log releases are filtered to the union of needed dates and target `uid`s,
then grouped by `(uid, date)`. Negative counts are clipped to zero; nonfinite
seconds become zero, and daily sums are kept in compact types. `rows` counts raw
records for the daily group. A date with a record is active even if every listening
measure is zero. The lazy Polars scan runs only when `collect(engine="streaming")`
materializes the grouped user-day table; previews below reuse that result.

In [22]:
def build_user_day_logs(parquet_dir, user_index, start, end):
    """One row per ``(uid, date)`` with the summed listening columns and the raw row count.

    ``start`` / ``end`` are ``YYYYMMDD`` ints. This is the single intermediate table that feeds
    both the monthly aggregates and the daily sequences.
    """
    parquet_dir = Path(parquet_dir)
    paths = [parquet_dir / "user_logs.parquet"]
    if _date_int(end) >= 20170301:
        march = parquet_dir / "user_logs_v2.parquet"
        if not march.exists():
            raise FileNotFoundError("The March cutoff requires user_logs_v2.parquet.")
        paths.append(march)
    lf = (pl.concat([pl.scan_parquet(path) for path in paths])
          .filter((pl.col("date") >= _date_int(start)) & (pl.col("date") <= _date_int(end)))
          .join(user_index.lazy().select("msno", "uid"), on="msno", how="inner")
          .group_by("uid", "date")
          .agg([pl.col(c).cast(pl.Int64).clip(lower_bound=0).sum()
                .clip(upper_bound=np.iinfo(np.int32).max).cast(pl.Int32)
                for c in LOG_COLS if c != "total_secs"]
               + [pl.when(pl.col("total_secs").is_finite())
                    .then(pl.col("total_secs")).otherwise(0).clip(lower_bound=0)
                    .sum().cast(pl.Float32), pl.len().alias("rows")]))
    return lf.collect(engine="streaming")

In [23]:
# One user-day table covers the earliest monthly window (February cohort) to the last cutoff.
LOG_START = min(window_start(cutoff) for cutoff in CUTOFF.values())
user_day = build_user_day_logs(PARQUET_DIR, user_index, LOG_START, CUTOFF['test'])
print('User-day rows:', user_day.height, '| from', LOG_START.date(), 'to', CUTOFF['test'])
print(user_day.head(3))


User-day rows: 72614379 | from 2016-11-01 to 2017-03-31
shape: (3, 10)
┌────────┬──────────┬────────┬────────┬───┬─────────┬─────────┬─────────────┬──────┐
│ uid    ┆ date     ┆ num_25 ┆ num_50 ┆ … ┆ num_100 ┆ num_unq ┆ total_secs  ┆ rows │
│ ---    ┆ ---      ┆ ---    ┆ ---    ┆   ┆ ---     ┆ ---     ┆ ---         ┆ ---  │
│ u32    ┆ i32      ┆ i32    ┆ i32    ┆   ┆ i32     ┆ i32     ┆ f32         ┆ u32  │
╞════════╪══════════╪════════╪════════╪═══╪═════════╪═════════╪═════════════╪══════╡
│ 708008 ┆ 20170319 ┆ 9      ┆ 2      ┆ … ┆ 9       ┆ 19      ┆ 2539.221924 ┆ 1    │
│ 954206 ┆ 20170330 ┆ 0      ┆ 0      ┆ … ┆ 10      ┆ 8       ┆ 2722.163086 ┆ 1    │
│ 464140 ┆ 20170318 ┆ 4      ┆ 1      ┆ … ┆ 23      ┆ 21      ┆ 6544.352051 ┆ 1    │
└────────┴──────────┴────────┴────────┴───┴─────────┴─────────┴─────────────┴──────┘


## 8. Listening features and parameters

Valid ages/modes and the 99.9th percentile clipping values come from the **February
cohort** and its window, so the March holdout never influences preprocessing. Monthly listening groups by user and calendar month;
the monthly mean is clipped at the daily-derived threshold, and the sum is
rebuilt from that mean times the record count. Ratios use total songs where
positive. `last`, `last_second` and `last_third` mean the most recent **active**
months among the three eligible months; they need not be consecutive.

`days_since_last_log` is relative to the split cutoff. A user without logs gets
`has_listening_logs=0` and an out-of-window recency sentinel. Daily v3 tensors
remain unscaled log1p values. Model experiments fit their own scales using fit
data. The aggregate clipping parameters here are not refitted inside each model
fold; that inherited limitation remains in this readability refactor.

In [24]:
def fit_preprocess_params(members, user_day, train_users, train_cutoff):
    """Learn imputation, clipping and sequence standardization from the training window only."""
    members_train = members.join(train_users.select("uid"), on="uid", how="semi")

    age_mean = float(
        members_train.filter((pl.col("bd") > 0) & (pl.col("bd") <= 100))
                     .select(pl.col("bd").mean()).item())
    via_mode = int(
        members_train.filter(pl.col("registered_via") >= 0)
                     .select(pl.col("registered_via").mode().first()).item())

    start, cutoff = _date_int(sequence_start(train_cutoff)), _date_int(train_cutoff)
    window = (user_day.join(train_users.select("uid"), on="uid", how="semi")
                       .filter((pl.col("date") >= start) & (pl.col("date") <= cutoff)))
    if window.is_empty():
        raise ValueError("No training listening records inside the sequence window.")

    clip, mean, std = {}, np.empty(len(LOG_COLS)), np.empty(len(LOG_COLS))
    for i, col in enumerate(LOG_COLS):
        upper = window.select(pl.col(col).quantile(0.999, interpolation="linear")).item()
        cell = window.select(pl.col(col).clip(upper_bound=upper).log1p()).to_series()
        clip[col] = float(upper)
        mean[i] = float(cell.mean())
        std[i] = float(cell.std(ddof=0))
    if not np.isfinite(mean).all() or not np.isfinite(std).all():
        raise ValueError("Non-finite daily sequence preprocessing parameters.")
    std[std < 1e-6] = 1.0

    return {
        "age_mean": age_mean,
        "registered_via_mode": via_mode,
        "log_clip_p999": clip,
        "seq_days": int(SEQ_DAYS),
        "seq_mean": mean.tolist(),
        "seq_std": std.tolist(),
    }


def save_preprocess_params(params, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(params, f, indent=2)


def load_preprocess_params(path):
    with open(path) as f:
        return json.load(f)

In [25]:
_LOG_TEMPLATES = {}
for _name in LOG_NAMES.values():
    _LOG_TEMPLATES[f"sum_{_name}"] = "sum_{p}_month_" + _name
    _LOG_TEMPLATES[f"mean_{_name}"] = "mean_{p}_month_" + _name
_LOG_TEMPLATES.update({
    "count": "{p}_login",
    "last_date": "{p}_login_date_last",
    "sum_total_songs": "{p}_sum_total_songs",
    "ratio_100": "{p}_ratio_100",
    "ratio_25": "{p}_ratio_25",
})
_PREFIX = {1: "last", 2: "last_second", 3: "last_third"}


def _add_relative_diffs(df):
    """Difference between the last and the second-to-last active month."""
    exprs = []
    for how in ("sum", "mean"):
        for metric in ("25", "50", "75", "985", "100", "unq", "total_secs"):
            exprs.append((pl.col(f"{how}_last_month_{metric}")
                          - pl.col(f"{how}_last_second_month_{metric}"))
                         .alias(f"{how}_diff_last_2_month_{metric}"))
    exprs.append((pl.col("last_login") - pl.col("last_second_login")).alias("login_difference"))
    return df.with_columns(exprs)


def build_log_features(user_day, users, cutoff, params):
    """Monthly log aggregates for the users' last / second / third active month (D-1)."""
    clip = params["log_clip_p999"]
    last_period = pd.Timestamp(cutoff).to_period("M")
    months = [int((last_period - offset).year * 100 + (last_period - offset).month)
              for offset in (2, 1, 0)]

    md = (user_day.join(users.select("uid"), on="uid", how="semi")
          .with_columns((pl.col("date") // 100).cast(pl.Int32).alias("month"))
          .group_by("uid", "month")
          .agg([pl.col(c).sum().alias(f"sum_{LOG_NAMES[c]}") for c in LOG_COLS]
               + [pl.col("rows").sum().cast(pl.Int64).alias("count"),
                  pl.col("date").max().alias("last_date")]))

    # The p99.9 is learned on daily values, so it is applied on the per-day mean; the monthly sum
    # is rebuilt from the clipped mean to keep the outlier control of the original pipeline.
    md = md.with_columns([
        (pl.col(f"sum_{LOG_NAMES[c]}").cast(pl.Float64) / pl.col("count"))
        .clip(upper_bound=clip[c]).alias(f"mean_{LOG_NAMES[c]}") for c in LOG_COLS])
    md = md.with_columns([
        (pl.col(f"mean_{n}") * pl.col("count")).round().cast(pl.Int64).alias(f"sum_{n}")
        for n in LOG_NAMES.values()])
    md = md.with_columns(
        pl.sum_horizontal([pl.col(f"sum_{n}") for n in ("25", "50", "75", "985", "100")])
          .alias("sum_total_songs"))
    songs = pl.when(pl.col("sum_total_songs") > 0).then(pl.col("sum_total_songs")).otherwise(None)
    md = md.with_columns((pl.col("sum_100") / songs).alias("ratio_100"),
                         (pl.col("sum_25") / songs).alias("ratio_25"))

    md = md.filter(pl.col("month").is_in(months))
    md = md.with_columns(
        pl.col("month").rank("ordinal", descending=True).over("uid").cast(pl.Int8).alias("rank"))

    parts = []
    for rank, prefix in _PREFIX.items():
        parts.append(
            md.filter(pl.col("rank") == rank).select(
                [pl.col("uid")]
                + [pl.col(metric).alias(template.format(p=prefix))
                   for metric, template in _LOG_TEMPLATES.items()]))

    log = parts[0]
    for part in parts[1:]:
        log = log.join(part, on="uid", how="full", coalesce=True)
    log = log.drop([c for c in ("last_second_login_date_last", "last_third_login_date_last")
                    if c in log.columns])
    log = log.with_columns(pl.col("last_login_date_last").fill_null(0).cast(pl.Int64))
    log = log.fill_null(0)
    return _add_relative_diffs(log)

In [26]:
# Fit on the training cohort only (February); March is the holdout and April the test.
params = fit_preprocess_params(members, user_day, users_by_split['train_feb'], CUTOFF['train_feb'])
print('Age mean:', round(params['age_mean'], 2),
      '| registered-via mode:', params['registered_via_mode'])
print('Daily clipping limits:', params['log_clip_p999'])


Age mean: 29.97 | registered-via mode: 7
Daily clipping limits: {'num_25': 146.0, 'num_50': 44.0, 'num_75': 18.0, 'num_985': 23.0, 'num_100': 334.0, 'num_unq': 238.0, 'total_secs': 78007.4296875}


In [27]:
# Inspect raw age/source values beside the actual cleaned member columns.
member_examples = (members.join(users_by_split['train_feb'].select('uid'), on='uid', how='semi')
                   .filter((pl.col('bd') <= 0) | (pl.col('bd') > 100) |
                           (pl.col('registered_via') < 0)).head(3))
cleaned_members = build_member_features(members, users_by_split['train_feb'], params,
                                        CUTOFF['train_feb'])
print(member_examples.select('uid', 'bd', 'registered_via').join(
    cleaned_members.select('uid', pl.col('bd').alias('clean_bd'), 'bd_missing',
                           pl.col('registered_via').alias('clean_registered_via')),
    on='uid', how='left'))


shape: (3, 6)
┌────────┬─────┬────────────────┬───────────┬────────────┬──────────────────────┐
│ uid    ┆ bd  ┆ registered_via ┆ clean_bd  ┆ bd_missing ┆ clean_registered_via │
│ ---    ┆ --- ┆ ---            ┆ ---       ┆ ---        ┆ ---                  │
│ u32    ┆ i16 ┆ i8             ┆ f32       ┆ i8         ┆ i16                  │
╞════════╪═════╪════════════════╪═══════════╪════════════╪══════════════════════╡
│ 503810 ┆ 0   ┆ 7              ┆ 29.971142 ┆ 1          ┆ 7                    │
│ 322917 ┆ 0   ┆ 7              ┆ 29.971142 ┆ 1          ┆ 7                    │
│ 732211 ┆ 0   ┆ 7              ┆ 29.971142 ┆ 1          ┆ 7                    │
└────────┴─────┴────────────────┴───────────┴────────────┴──────────────────────┘


## 9. Final user table

Start with one row per target user, then left join member, transaction and listening
features separately on `uid`. Left joins preserve users without member or log
records. The expected cardinality is at most one feature row per `uid`; the checks
below catch accidental row multiplication. Null feature values become zero,
types shrink, and `row` restores source order. Only the labelled cohorts join `is_churn`.

| Final feature | Source | Transformation |
|---|---|---|
| `bd`, `bd_missing` | members | valid age or train mean; invalid flag |
| `transaction_count`, `days_to_expire` | transactions | count; last expiry minus cutoff |
| `last_sum_total_songs` | user-day logs | clipped sum in latest active month |
| `days_since_last_log`, `has_listening_logs` | user-day logs | recency and presence |

`metadata.json` lists the complete final feature catalog.

In [28]:
example_users = pl.DataFrame({'uid': [1, 2]})
example_members = pl.DataFrame({'uid': [1], 'city': [4]})
print('Left join keeps a user without a member row:')
print(example_users.join(example_members, on='uid', how='left'))

Left join keeps a user without a member row:
shape: (2, 2)
┌─────┬──────┐
│ uid ┆ city │
│ --- ┆ ---  │
│ i64 ┆ i64  │
╞═════╪══════╡
│ 1   ┆ 4    │
│ 2   ┆ null │
└─────┴──────┘


In [29]:
def _shrink(df):
    casts = [pl.col(name).cast(pl.Float32) if dtype == pl.Float64 else
             pl.col(name).cast(pl.Int32) if dtype == pl.Int64 else None
             for name, dtype in df.schema.items()]
    return df.with_columns([c for c in casts if c is not None])


def relative_log_recency(frame, cutoff):
    """Replace a literal listening date with age relative to the prediction cutoff."""
    last_date = pl.col("last_login_date_last").cast(pl.String).str.to_date("%Y%m%d", strict=False)
    sentinel = max(SEQ_DAYS, (pd.Timestamp(cutoff) - window_start(cutoff)).days + 1) + 1
    return (frame.with_columns(
        pl.when(last_date.is_not_null())
          .then((pl.lit(_date(cutoff)) - last_date).dt.total_days())
          .otherwise(sentinel).cast(pl.Int32).alias("days_since_last_log"),
        last_date.is_not_null().cast(pl.Int8).alias("has_listening_logs"),
    ).drop("last_login_date_last"))


def build_features(members, trans, user_day, users, cutoff, params):
    """Build the full feature set for one set, in ``users`` (row) order."""
    member_feature = build_member_features(members, users, params, cutoff)
    transaction_feature = build_transaction_features(trans, users, cutoff)
    log_feature = build_log_features(user_day, users, cutoff, params)

    combined = (users.select("uid", "msno", "row")
                .join(member_feature, on="uid", how="left", maintain_order="left")
                .join(transaction_feature, on="uid", how="left", maintain_order="left")
                .join(log_feature, on="uid", how="left", maintain_order="left"))
    combined = relative_log_recency(combined, cutoff)

    features = [c for c in combined.columns if c not in ("uid", "msno", "row")]
    combined = combined.with_columns([pl.col(c).fill_null(0) for c in features])
    combined = _shrink(combined).sort("row")
    return combined.select(["msno"] + features)

In [30]:
feature_tables = {}
for split in SPLITS:
    users, cutoff = users_by_split[split], CUTOFF[split]
    member_part = build_member_features(members, users, params, cutoff)
    transaction_part = build_transaction_features(transactions, users, cutoff)
    listening_part = build_log_features(user_day, users, cutoff, params)
    print(split, 'join inputs:', member_part.shape, transaction_part.shape, listening_part.shape)
    assert all(part['uid'].n_unique() == part.height for part in
               (member_part, transaction_part, listening_part))
    joined = users.select('uid', 'msno', 'row')
    for part in (member_part, transaction_part, listening_part):
        joined = joined.join(part, on='uid', how='left', maintain_order='left')
        assert joined.height == users.height
    joined = relative_log_recency(joined, cutoff)
    feature_names = [c for c in joined.columns if c not in ('uid', 'msno', 'row')]
    joined = joined.with_columns([pl.col(c).fill_null(0) for c in feature_names])
    feature_tables[split] = _shrink(joined).sort('row').select(['msno'] + feature_names)

final_tables = {}
for split in SPLITS:
    keys = (labelled_users[split].select('msno', 'is_churn') if split in labelled_users
            else test_users.select('msno'))
    final_tables[split] = keys.join(feature_tables[split], on='msno', how='left', maintain_order='left')
schemas = {split: final_tables[split].drop('is_churn', strict=False).schema for split in SPLITS}
assert all(s == schemas['train'] for s in schemas.values()), 'cohort tables differ in columns/types'
print('Final tables:', {split: frame.shape for split, frame in final_tables.items()})
print(final_tables['train_feb'].select('msno', 'is_churn', 'bd', 'transaction_count',
                                       'days_to_expire', 'has_listening_logs').head(3))


train_feb join inputs: (877161, 8) (990836, 29) (808121, 71)
train join inputs: (860967, 8) (968436, 29) (793729, 71)
test join inputs: (795090, 8) (907471, 29) (733064, 71)
Final tables: {'train_feb': (992931, 108), 'train': (970960, 108), 'test': (907471, 107)}
shape: (3, 6)
┌──────────────────────┬──────────┬──────┬───────────────────┬────────────────┬────────────────────┐
│ msno                 ┆ is_churn ┆ bd   ┆ transaction_count ┆ days_to_expire ┆ has_listening_logs │
│ ---                  ┆ ---      ┆ ---  ┆ ---               ┆ ---            ┆ ---                │
│ str                  ┆ i8       ┆ f32  ┆ u32               ┆ i32            ┆ i8                 │
╞══════════════════════╪══════════╪══════╪═══════════════════╪════════════════╪════════════════════╡
│ waLDQMmcOu2jLDaV1ddD ┆ 1        ┆ 36.0 ┆ 2                 ┆ 6              ┆ 1                  │
│ kgCrB/jl6s…          ┆          ┆      ┆                   ┆                ┆                    │
│ QA7uiXy8vIbUS

## 10. Tabular export

The compact tabular contract is published last: `train_feb.parquet`, `train.parquet` and
`test.parquet` plus `preprocess_params.json` and `metadata.json`. Metadata records cutoffs,
label months, feature names, row counts and `msno` order hashes per cohort. The checks verify
source order, uniqueness, absence of nulls, finite floats and the row hash. Metadata is written
after every table exists.


In [31]:
def export_tabular_dataset(export_dir, tables, metadata):
    """Write one parquet per cohort (``tables``: name -> frame) and their contract."""
    export_dir = Path(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)
    # Invalidate a previous manifest before overwriting any part of this export.
    (export_dir / "metadata.json").unlink(missing_ok=True)

    for name, frame in tables.items():
        frame.write_parquet(export_dir / f"{name}.parquet", compression="zstd")

    metadata = dict(metadata)
    metadata["dataset_version"] = DATASET_VERSION
    metadata["splits"] = list(tables)
    metadata["features"] = [c for c in tables["train"].columns if c not in ("msno", "is_churn")]
    for name, frame in tables.items():
        metadata[f"n_{name}"] = int(frame.height)
        metadata[f"hash_{name}"] = _hash_msno(frame)
        if "is_churn" in frame.columns:
            metadata[f"churn_rate_{name}"] = float(frame["is_churn"].mean())
    metadata["generated_at"] = pd.Timestamp.now().isoformat(timespec="seconds")
    with open(export_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2, default=str)
    return metadata


In [32]:
save_preprocess_params(params, EXPORT_DIR / 'preprocess_params.json')
metadata = export_tabular_dataset(EXPORT_DIR, final_tables, {
    'cutoffs': CUTOFF, 'label_months': LABEL_MONTHS, 'label_files': LABEL_FILES,
    'categorical': CATEGORICAL, 'preprocess_params': params,
    'preprocess_params_fit_on': 'train_feb', 'sample_users': SAMPLE_USERS,
})
for split, frame in final_tables.items():
    source = source_users[split]
    assert frame.height == source.height
    assert frame['msno'].to_list() == source['msno'].to_list()
    assert frame['msno'].n_unique() == frame.height
    assert frame.null_count().sum_horizontal().item() == 0
    float_columns = [c for c in metadata['features'] if frame.schema[c].is_float()]
    if float_columns:
        assert frame.select(pl.all_horizontal([pl.col(c).is_finite() for c in float_columns])).to_series().all()
    assert metadata[f'hash_{split}'] == _hash_msno(frame)
if SAMPLE_USERS is None:
    assert 0.05 < final_tables['train_feb']['is_churn'].mean() < 0.08
    assert 0.08 < final_tables['train']['is_churn'].mean() < 0.10
print('Export:', EXPORT_DIR.resolve())
for split in LABEL_FILES:
    print(f'{split} churn rate:', round(float(final_tables[split]['is_churn'].mean()), 4))
# Shift between the training and holdout cohorts on the strongest timing feature.
for split in LABEL_FILES:
    q = final_tables[split]['days_to_expire'].quantile
    print(f'{split} days_to_expire p1/p50/p99:', q(0.01), q(0.5), q(0.99))
for path in sorted(EXPORT_DIR.iterdir()):
    if path.is_file():
        print(path.name, round(path.stat().st_size / 1e6, 2), 'MB')
print('Peak RAM (reported by OS):', round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2), 'GB')


Export: /home/alejandropenagos/Escritorio/Alejandro/kkbox-churn-prediction-challenge/workshops/w1/export-tabular-v4
train_feb churn rate: 0.0639
train churn rate: 0.0899
train_feb days_to_expire p1/p50/p99: -6.0 17.0 56.0
train days_to_expire p1/p50/p99: -7.0 18.0 61.0
metadata.json 0.0 MB
preprocess_params.json 0.0 MB
test.parquet 163.71 MB
train.parquet 174.0 MB
train_feb.parquet 178.3 MB
Peak RAM (reported by OS): 14.58 GB
